
# Incremental Load in Azure Data Factory (ADF)

## What is Incremental Load?

An **Incremental Load** is a data loading technique where **only new or modified records** are loaded from the source system to the destination, instead of loading the entire dataset every time.

### Why use Incremental Load?

Suppose your source table has **10 million records**.

Every day, only **10,000 records** are added or updated.

### Without Incremental Load

```text
Day 1 → Load 10,000,000 rows

Day 2 → Load 10,010,000 rows

Day 3 → Load 10,020,000 rows
```

This is inefficient because millions of unchanged rows are processed repeatedly.

### With Incremental Load

```text
Day 1 → Load 10,000,000 rows

Day 2 → Load only 10,000 new rows

Day 3 → Load only 10,000 new rows
```

This approach:

* Improves performance
* Reduces execution time
* Lowers compute costs
* Minimizes network traffic

---

# Types of Incremental Load

There are two common approaches:

1. **Incremental Load using Watermark**
2. **Incremental Load without Watermark (Change Tracking or Full Comparison)**

---

# 1. Incremental Load Using Watermark

## What is a Watermark?

A **watermark** is a value that indicates the **last successfully processed record**.

Common watermark columns include:

* LastModifiedDate
* UpdatedDate
* CreatedDate
* OrderID
* TransactionID
* Identity column

Example:

| CustomerID | Name  | LastModified |
| ---------- | ----- | ------------ |
| 1          | John  | 2026-07-10   |
| 2          | David | 2026-07-12   |
| 3          | Mike  | 2026-07-14   |

If the last successful load processed records up to **2026-07-12**, the watermark is:

```text
2026-07-12
```

The next run loads:

```sql
SELECT *
FROM Customers
WHERE LastModified > '2026-07-12';
```

Only the new or updated records are copied.

---

## Real-Time Architecture

```text
Source SQL
      │
      ▼
Lookup Activity
(Read Last Watermark)
      │
      ▼
Copy Activity
WHERE LastModified > Watermark
      │
      ▼
Azure SQL / ADLS
      │
      ▼
Stored Procedure
(Update Watermark)
```

---

## Step-by-Step Process

### Step 1 – Create a Watermark Table

```sql
CREATE TABLE WatermarkTable
(
    TableName NVARCHAR(100),
    LastWatermark DATETIME
);
```

Sample data:

| TableName | LastWatermark |
| --------- | ------------- |
| Customers | 2026-07-12    |

---

### Step 2 – Lookup Activity

Read the watermark:

```sql
SELECT LastWatermark
FROM WatermarkTable
WHERE TableName='Customers';
```

Suppose it returns:

```text
2026-07-12
```

---

### Step 3 – Copy Activity

Use a parameterized query:

```sql
SELECT *
FROM Customers
WHERE LastModified > '@{activity('Lookup').output.firstRow.LastWatermark}'
```

Only new records are loaded.

---

### Step 4 – Update the Watermark

Find the latest processed timestamp:

```sql
SELECT MAX(LastModified)
FROM Customers;
```

Suppose it returns:

```text
2026-07-14
```

Update the watermark table:

```sql
UPDATE WatermarkTable
SET LastWatermark='2026-07-14'
WHERE TableName='Customers';
```

The next execution starts from this value.

---

# Complete Pipeline

```text
Lookup Watermark
      │
      ▼
Copy Activity
      │
      ▼
Stored Procedure
(Update Watermark)
```

---

# Real-Time Example

### Source Table

| ID | Name  | LastModified |
| -- | ----- | ------------ |
| 1  | John  | 10-Jul       |
| 2  | Mike  | 11-Jul       |
| 3  | David | 12-Jul       |

Watermark:

```text
12-Jul
```

Next day:

| ID | Name | LastModified |
| -- | ---- | ------------ |
| 4  | Sara | 13-Jul       |
| 5  | Alex | 14-Jul       |

ADF query:

```sql
SELECT *
FROM Customers
WHERE LastModified > '2026-07-12';
```

Only Sara and Alex are loaded.

---

# Advantages

* Fast
* Efficient
* Low cost
* Simple to implement
* Ideal when a reliable timestamp or sequence column exists

---

# 2. Incremental Load Without Watermark

Sometimes the source table **does not have**:

* LastModifiedDate
* UpdatedDate
* CreatedDate

or those columns are unreliable.

In that case, other techniques are used.

---

## Method 1 – Primary Key Comparison

### Source

| ID | Name  |
| -- | ----- |
| 1  | John  |
| 2  | David |
| 3  | Sara  |

### Destination

| ID | Name  |
| -- | ----- |
| 1  | John  |
| 2  | David |

Compare the tables:

```sql
SELECT *
FROM Source s
WHERE NOT EXISTS
(
SELECT 1
FROM Destination d
WHERE s.ID=d.ID
)
```

Only Sara is loaded.

---

## Method 2 – MERGE Statement

```sql
MERGE Target t
USING Source s
ON t.CustomerID=s.CustomerID

WHEN MATCHED THEN
UPDATE SET
Name=s.Name

WHEN NOT MATCHED THEN
INSERT
VALUES(...)
```

This handles both inserts and updates in one operation.

---

## Method 3 – Full Outer Join Comparison

```text
Source

↓

Full Outer Join

↓

Compare Records

↓

Insert

↓

Update
```

Used when no timestamp exists.

---

## Method 4 – Hash Comparison

Generate a hash for each row.

Example:

```text
CustomerID

Name

City

↓

SHA256 Hash
```

If the hash changes, the row has changed.

This is common in enterprise data warehouses.

---

# Real-Time Architecture (Without Watermark)

```text
Source

↓

Copy to Stage

↓

MERGE

↓

Target
```

No watermark table is required.

---

# Watermark vs No Watermark

| Feature                | With Watermark | Without Watermark |
| ---------------------- | -------------- | ----------------- |
| Uses LastModifiedDate  | ✅ Yes          | ❌ No              |
| Uses CreatedDate       | ✅ Yes          | ❌ No              |
| Needs timestamp column | ✅ Yes          | ❌ No              |
| Faster                 | ✅ Yes          | Usually slower    |
| Compares rows          | ❌ No           | ✅ Yes             |
| Enterprise preference  | ⭐⭐⭐⭐⭐          | ⭐⭐⭐               |

---

# Which Method Do Companies Use?

### Method 1 – Watermark (Most Common)

Used when tables have:

* LastModifiedDate
* UpdatedDate
* CreatedDate

Example:

```text
SQL Server

↓

ADF

↓

ADLS
```

---

### Method 2 – SQL Change Tracking / Change Data Capture (CDC)

The source database tracks inserts, updates, and deletes automatically.

ADF reads only those changes.

---

### Method 3 – MERGE

Common in Delta Lake and Azure Synapse to upsert data into target tables.

---

### Method 4 – Hash Comparison

Used when source systems don't provide timestamps or change tracking.

---

# Real Enterprise Example

A retail company receives daily customer updates.

```text
SQL Server
(Customer Table)
      │
      ▼
Lookup Activity
(Read Watermark)
      │
      ▼
Copy Activity
WHERE LastModified > Watermark
      │
      ▼
ADLS Bronze
      │
      ▼
Databricks
      │
      ▼
Delta Silver
      │
      ▼
MERGE INTO Gold Table
      │
      ▼
Stored Procedure
(Update Watermark)
```

---

# Interview Questions

### 1. What is Incremental Load?

**Answer:** Incremental load is a technique where only new or changed records are loaded from the source into the target, reducing processing time and resource usage.

---

### 2. What is a Watermark?

**Answer:** A watermark is the last successfully processed value (such as a timestamp or sequence number). It is stored separately and used to identify which records should be processed in the next pipeline run.

---

### 3. How do you implement Watermark Incremental Load in ADF?

**Answer:**

1. Store the last processed watermark in a control table.
2. Use a **Lookup Activity** to retrieve the watermark.
3. Use the watermark in the **Copy Activity** source query to filter new or updated records.
4. After a successful load, update the watermark table with the latest processed value using a **Stored Procedure Activity** or **Script Activity**.

---

### 4. What if there is no watermark column?

**Answer:** Alternatives include:

* SQL Change Tracking
* Change Data Capture (CDC)
* Primary key comparison
* `MERGE` statements
* Hash-based row comparison

These methods identify new or changed records without relying on a timestamp.

---

# Best Practice

For modern Azure Data Engineering projects, a common pattern is:

* **ADF** for orchestration
* **Watermark-based incremental ingestion** into ADLS Bronze
* **Databricks + Delta Lake MERGE** for upserts into Silver and Gold layers
* **Azure SQL control table** to store watermark values
* **Azure DevOps** to manage deployment and version control

This approach is scalable, efficient, and widely used in enterprise data platforms.


Absolutely. **Incremental loading without a watermark** is one of the most common interview topics because many legacy systems don't have a `LastModifiedDate` or `UpdatedDate` column.

In enterprise projects, there are **five major approaches**:

1. Primary Key Comparison
2. SQL MERGE (Upsert)
3. Change Data Capture (CDC)
4. SQL Change Tracking (CT)
5. Hash-Based Comparison

Let's go through each one with **real-world examples, ADF implementation, SQL examples, and when to use it**.

---

# Scenario

Suppose we have a legacy ERP system.

### Source Table

| CustomerID | Name  | City     |
| ---------- | ----- | -------- |
| 1          | John  | New York |
| 2          | David | Seattle  |
| 3          | Mike  | Chicago  |

There is **NO**

* UpdatedDate
* LastModifiedDate
* Timestamp

How do we know which records changed?

---

# Method 1 — Primary Key Comparison

## When is it used?

When:

* Source has Primary Key
* Destination already contains previous data
* Only **new records** need to be inserted

This is the simplest incremental load.

---

## Example

### Source

| ID | Name  |
| -- | ----- |
| 1  | John  |
| 2  | David |
| 3  | Mike  |
| 4  | Sara  |

Destination

| ID | Name  |
| -- | ----- |
| 1  | John  |
| 2  | David |
| 3  | Mike  |

Record **4** is new.

---

## SQL

```sql
SELECT *
FROM Source s
WHERE NOT EXISTS
(
SELECT 1
FROM Destination d
WHERE s.ID=d.ID
)
```

Returns

| ID | Name |
| -- | ---- |
| 4  | Sara |

Only Sara is inserted.

---

## ADF Pipeline

```text
Copy Source

↓

Stage Table

↓

Stored Procedure

↓

Insert New Records
```

---

## Real Company Example

Hospital system

Patients

```
PatientID
PatientName
```

Every day

* New patients are added
* Old patient records never change

Only insert new PatientIDs.

No watermark needed.

---

## Advantages

✅ Very simple

✅ Fast

---

## Limitation

Cannot detect updates.

---

# Method 2 — MERGE Statement (UPSERT)

This is the **most common enterprise solution**.

Instead of checking only inserts,

MERGE checks

* Inserts
* Updates

simultaneously.

---

## Example

Yesterday

Destination

| ID | Name  | City     |
| -- | ----- | -------- |
| 1  | John  | New York |
| 2  | David | Seattle  |

Today

Source

| ID | Name  | City    |
| -- | ----- | ------- |
| 1  | John  | Boston  |
| 2  | David | Seattle |
| 3  | Sara  | Dallas  |

Notice

John's city changed.

Sara is new.

MERGE handles both.

---

## SQL

```sql
MERGE Target AS T

USING Source AS S

ON T.ID=S.ID

WHEN MATCHED THEN

UPDATE

SET

T.Name=S.Name,

T.City=S.City

WHEN NOT MATCHED THEN

INSERT(ID,Name,City)

VALUES(S.ID,S.Name,S.City);
```

Result

Destination

| ID | Name  | City    |
| -- | ----- | ------- |
| 1  | John  | Boston  |
| 2  | David | Seattle |
| 3  | Sara  | Dallas  |

---

## ADF Flow

```text
Copy Activity

↓

Stage Table

↓

Stored Procedure

↓

MERGE
```

---

## Real Company Example

E-commerce

Customer Address changes frequently.

Need

* Insert new customers
* Update changed addresses

MERGE is perfect.

---

# Method 3 — SQL Change Data Capture (CDC)

This is SQL Server's built-in feature.

Instead of comparing tables,

SQL Server itself tracks

* Inserts

* Updates

* Deletes

---

## Example

Yesterday

```
John

David
```

Today

```
John Updated

David Deleted

Sara Inserted
```

CDC records

```
Update

Delete

Insert
```

Automatically.

---

## SQL

Instead of

```sql
SELECT *

FROM Customer
```

You read

```sql
cdc.fn_cdc_get_all_changes_Customer(...)
```

It returns

| Operation | Customer |
| --------- | -------- |
| Insert    | Sara     |
| Update    | John     |
| Delete    | David    |

---

## ADF Pipeline

```text
SQL CDC

↓

ADF

↓

Copy Activity

↓

ADLS
```

---

## Real Company Example

Bank

Every second

Thousands of transactions happen.

Need

Insert

Update

Delete

CDC captures all automatically.

---

## Advantages

No comparison needed.

---

## Limitation

Requires CDC enabled.

---

# Method 4 — SQL Change Tracking

This is lighter than CDC.

Instead of storing old/new values,

SQL only stores

"What changed?"

---

Example

Yesterday

```
John

David
```

Today

John changed.

SQL stores

```
CustomerID=1 Changed
```

ADF then fetches

CustomerID=1

---

SQL

```sql
CHANGETABLE(CHANGES Customer,@Version)
```

Returns

| CustomerID |
| ---------- |
| 1          |
| 5          |
| 8          |

ADF fetches these rows only.

---

## ADF Flow

```text
Read Change Tracking Version

↓

Fetch Changed IDs

↓

Copy Changed Records
```

---

## Real Company Example

CRM

Millions of customers.

Only

100 changed today.

Load only those 100.

---

## Advantages

Very fast.

Less storage than CDC.

---

# Method 5 — Hash Comparison

Very common in

* Data Warehouses
* Databricks
* Delta Lake

---

Suppose

Yesterday

| ID | Name | City     |
| -- | ---- | -------- |
| 1  | John | New York |

Generate

```
SHA256

↓

ABCD12345
```

Today

John moved.

| ID | Name | City   |
| -- | ---- | ------ |
| 1  | John | Boston |

Generate

```
SHA256

↓

XYZ98765
```

Hashes differ.

Therefore

Record changed.

---

## SQL Example

```sql
SELECT

CustomerID,

HASHBYTES

(

'SHA2_256',

CONCAT(Name,City)

)

AS HashValue

FROM Customer
```

---

Destination

| ID | Hash    |
| -- | ------- |
| 1  | ABCD123 |

Source

| ID | Hash   |
| -- | ------ |
| 1  | XYZ456 |

Compare

Hashes differ

↓

Update record.

---

## ADF Flow

```text
Copy Source

↓

Generate Hash

↓

Compare Hash

↓

MERGE
```

---

## Real Company Example

Insurance

Customer

* Name

* Address

* Phone

Any field changes

↓

Hash changes

↓

Update record.

---

# Method 6 — Full Outer Join Comparison

Sometimes companies cannot enable CDC or Change Tracking.

They simply compare

Source

and

Destination.

---

Yesterday

Destination

| ID | Name  |
| -- | ----- |
| 1  | John  |
| 2  | David |

Today

Source

| ID | Name          |
| -- | ------------- |
| 1  | John          |
| 2  | David Updated |
| 3  | Sara          |

Perform

```sql
SELECT *
FROM Source s
FULL OUTER JOIN Destination d
ON s.ID=d.ID
```

Logic

If

* Exists only in Source → Insert
* Exists in both but values differ → Update
* Exists only in Destination → Delete (optional)

---

# Which Method is Used in Real Projects?

| Method                     | Detect Inserts | Detect Updates | Detect Deletes       | Performance | Common Usage                            |
| -------------------------- | -------------- | -------------- | -------------------- | ----------- | --------------------------------------- |
| Primary Key Comparison     | ✅              | ❌              | ❌                    | ⭐⭐⭐⭐⭐       | Static reference data                   |
| MERGE                      | ✅              | ✅              | Optional             | ⭐⭐⭐⭐        | Data warehouses, Delta Lake             |
| Change Data Capture (CDC)  | ✅              | ✅              | ✅                    | ⭐⭐⭐⭐⭐       | SQL Server OLTP systems                 |
| Change Tracking            | ✅              | ✅              | ❌ (change info only) | ⭐⭐⭐⭐⭐       | High-volume SQL Server systems          |
| Hash Comparison            | ✅              | ✅              | ❌                    | ⭐⭐⭐⭐        | Databricks, Delta Lake, Data Warehouses |
| Full Outer Join Comparison | ✅              | ✅              | ✅                    | ⭐⭐          | Legacy systems, small/medium datasets   |

# Enterprise Recommendation

In most modern Azure Data Engineering projects, you'll typically see one of these patterns:

* **SQL Server with CDC** → ADF reads only changed rows → ADLS Bronze → Databricks → Delta Lake `MERGE` into Silver.
* **Legacy databases without timestamps** → Copy source to a staging table → Use SQL `MERGE` to apply inserts and updates.
* **Large data lake environments (Databricks/Delta Lake)** → Compute hashes for business columns and use Delta Lake `MERGE` to identify and apply changes efficiently.

These are the approaches interviewers most often expect you to understand for real-world Azure Data Factory and Databricks implementations.
